# CST-POC · Reasoning-Level LM — Colab Trainer

Train the tiny GPT causal LM over the **reasoning-token stream** of the
cst-poc corpus on a Colab GPU. Self-contained notebook — no git clone,
no CAMeL/spaCy needed. You just upload the already-tokenized JSONL +
reasoning vocab to Google Drive once.

**Inputs (upload to Drive before running):**
- `stage-2a-prop_logic.tokenized.jsonl`
- `stage-2b-syllogisms.tokenized.jsonl`
- `stage-2c-algebra_engine.tokenized.jsonl`
- `vocab-reasoning.json`

**Output:** `ckpt.pt` saved back to Drive.

**Runtime:** T4 → ~5-15 min; A100 → ~1-3 min for the full 3-epoch run.


## 1 · GPU check

In [ ]:
!nvidia-smi || echo "No GPU — set Runtime → Change runtime type → GPU"
import torch
print("torch", torch.__version__, "| cuda?", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 2 · Mount Drive & locate data

Default layout expected in your Drive:

```
MyDrive/cst-poc/
  reasoning/
    tokenized/
      stage-2a-prop_logic.tokenized.jsonl
      stage-2b-syllogisms.tokenized.jsonl
      stage-2c-algebra_engine.tokenized.jsonl
      vocab-reasoning.json
    runs/                       # ← checkpoints go here
```

Adjust `DATA_DIR` / `RUN_NAME` if your layout differs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DATA_DIR  = Path("/content/drive/MyDrive/cst-poc/reasoning/tokenized")
RUNS_DIR  = Path("/content/drive/MyDrive/cst-poc/reasoning/runs")
RUN_NAME  = "colab-v0"
OUT_DIR   = RUNS_DIR / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Which stages to train on. Comment out whichever you want to skip.
DATA_FILES = [
    DATA_DIR / "stage-2a-prop_logic.tokenized.jsonl",
    DATA_DIR / "stage-2b-syllogisms.tokenized.jsonl",
    DATA_DIR / "stage-2c-algebra_engine.tokenized.jsonl",
]
VOCAB_PATH = DATA_DIR / "vocab-reasoning.json"

for p in DATA_FILES + [VOCAB_PATH]:
    assert p.exists(), f"Missing {p}"
    print(f"  ✓ {p.name}  ({p.stat().st_size/1e6:.1f} MB)")

## 3 · Model definition (`TinyGPT`)

Inline copy of `reasoning/train/model.py` — ~5-15 M params depending on
config. Weight-tied head, pre-norm blocks, learned positional embeddings.

In [ ]:
import math
from dataclasses import dataclass
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class GPTConfig:
    vocab_size: int
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    d_ff: int = 1536
    max_len: int = 256
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        self.n_heads = cfg.n_heads
        self.d_head = cfg.d_model // cfg.n_heads
        self.qkv  = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.drop = nn.Dropout(cfg.dropout)
        self.register_buffer("mask",
            torch.tril(torch.ones(cfg.max_len, cfg.max_len, dtype=torch.bool)),
            persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_head)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        att = att.masked_fill(~self.mask[:T, :T], float("-inf"))
        att = self.drop(F.softmax(att, dim=-1))
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout),
        )
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_len, cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight    # weight tying
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, ids):
        B, T = ids.shape
        pos = torch.arange(T, device=ids.device).unsqueeze(0).expand(B, T)
        x = self.drop(self.tok_emb(ids) + self.pos_emb(pos))
        for blk in self.blocks: x = blk(x)
        return self.head(self.ln_f(x))

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters())

## 4 · Dataset

Load tokenized records from the JSONL stage files, concatenate
`question + cot + answer` reasoning-token streams, map through vocab,
pad to `max_len`. Answer kept at the tail when we truncate.

In [ ]:
import json, random
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset

def load_vocab(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def ids_from_tokens(tokens, vocab):
    unk = vocab["[UNK]"]
    return [vocab.get(t, unk) for t in tokens]

def _flatten(rec, vocab):
    seq = list(rec["question_tokens"]["reasoning"])
    for step in rec["cot_tokens"]:
        seq.extend(step["reasoning"])
    seq.extend(rec["answer_tokens"]["reasoning"])
    return ids_from_tokens(seq, vocab)

class ReasoningJsonlDataset(Dataset):
    def __init__(self, path, vocab, max_len=256):
        self.vocab = vocab
        self.max_len = max_len
        self.pad_id = vocab["[PAD]"]
        self._seqs = []
        with open(path, "r", encoding="utf-8") as fh:
            for line in fh:
                ids = _flatten(json.loads(line), vocab)
                if len(ids) > max_len:
                    ids = ids[-max_len:]           # keep answer at tail
                self._seqs.append(ids)

    def __len__(self): return len(self._seqs)
    def __getitem__(self, i):
        seq = self._seqs[i]
        x = torch.full((self.max_len,), self.pad_id, dtype=torch.long)
        x[: len(seq)] = torch.tensor(seq, dtype=torch.long)
        return {"ids": x, "length": torch.tensor(len(seq), dtype=torch.long)}

def collate(batch):
    ids = torch.stack([b["ids"] for b in batch], dim=0)
    lengths = torch.stack([b["length"] for b in batch], dim=0)
    inputs  = ids[:, :-1]
    targets = ids[:, 1:].clone()
    for i, L in enumerate(lengths.tolist()):
        if L - 1 < targets.size(1):
            targets[i, L - 1:] = -100       # ignore padding in CE
    return {"inputs": inputs, "targets": targets, "lengths": lengths}

def split_indices(n, val_frac=0.05, seed=42):
    rng = random.Random(seed)
    idxs = list(range(n)); rng.shuffle(idxs)
    n_val = max(1, int(n * val_frac))
    return idxs[n_val:], idxs[:n_val]

## 5 · Build dataset + model

Loads every enabled stage file, concats, splits into train/val, and
instantiates `TinyGPT`. Also keeps `stage_slices` so we can run a
syllogism-only eval at the end.

In [ ]:
# ── Hyperparameters ────────────────────────────────────────
MAX_LEN   = 256
BATCH     = 64
EPOCHS    = 3
LR        = 3e-4
WD        = 0.01
SEED      = 42

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vocab = load_vocab(VOCAB_PATH)
print(f"vocab size: {len(vocab)}")

per_stage = []
for f in DATA_FILES:
    ds = ReasoningJsonlDataset(f, vocab, max_len=MAX_LEN)
    print(f"  {f.name}: {len(ds)} records")
    per_stage.append(ds)

full_ds = ConcatDataset(per_stage)
train_idx, val_idx = split_indices(len(full_ds), val_frac=0.05, seed=SEED)
train_ds = Subset(full_ds, train_idx)
val_ds   = Subset(full_ds, val_idx)
print(f"train={len(train_ds)}  val={len(val_ds)}")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True,
                          collate_fn=collate, drop_last=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True,
                          collate_fn=collate, persistent_workers=True)

cfg = GPTConfig(vocab_size=len(vocab), max_len=MAX_LEN)
model = TinyGPT(cfg).to(device)
print(f"model params: {model.num_parameters()/1e6:.2f} M on {device}")

## 6 · Train (AMP + cosine LR)

Mixed-precision causal-LM loss, AdamW, cosine schedule with 500 warmup
steps, grad clip 1.0. Checkpoints to Drive at the end of every epoch.

In [ ]:
import time
from tqdm.auto import tqdm

total_steps = len(train_loader) * EPOCHS
warmup = min(500, total_steps // 20)

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD, betas=(0.9, 0.95))

def lr_at(step):
    if step < warmup:
        return step / max(1, warmup)
    prog = (step - warmup) / max(1, total_steps - warmup)
    return 0.5 * (1 + math.cos(math.pi * prog))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

@torch.no_grad()
def evaluate():
    model.eval()
    tot, tok = 0.0, 0
    for b in val_loader:
        ids = b["inputs"].to(device, non_blocking=True)
        tgt = b["targets"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda"), dtype=torch.float16):
            logits = model(ids)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                               tgt.reshape(-1), ignore_index=-100, reduction="sum")
        tot += loss.item()
        tok += (tgt != -100).sum().item()
    m = tot / max(1, tok)
    return m, math.exp(min(m, 20))

log_path = OUT_DIR / "train_log.jsonl"
log_fh = open(log_path, "w", encoding="utf-8")

step, t0 = 0, time.time()
for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}", leave=False)
    for b in pbar:
        ids = b["inputs"].to(device, non_blocking=True)
        tgt = b["targets"].to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=(device.type == "cuda"), dtype=torch.float16):
            logits = model(ids)
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                   tgt.reshape(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
        step += 1
        if step % 100 == 0:
            lr = opt.param_groups[0]["lr"]
            pbar.set_postfix(loss=f"{loss.item():.3f}", lr=f"{lr:.1e}")
            log_fh.write(json.dumps({"step": step, "epoch": epoch,
                                     "train_loss": loss.item(), "lr": lr,
                                     "elapsed_s": round(time.time() - t0, 1)}) + "\n")
            log_fh.flush()
    vl, vp = evaluate()
    print(f"── epoch {epoch+1}  val_loss={vl:.4f}  val_ppl={vp:.2f}  "
          f"({time.time()-t0:.0f}s)")
    log_fh.write(json.dumps({"step": step, "epoch": epoch,
                             "val_loss": vl, "val_ppl": vp,
                             "elapsed_s": round(time.time() - t0, 1)}) + "\n")
    log_fh.flush()
    # Mid-run checkpoint
    torch.save({"state_dict": model.state_dict(),
                "config": cfg.__dict__,
                "vocab_size": len(vocab),
                "epoch": epoch},
               OUT_DIR / f"ckpt-epoch{epoch+1}.pt")

log_fh.close()
print(f"done in {time.time()-t0:.0f}s")

## 7 · Save final checkpoint to Drive

In [ ]:
ckpt_path = OUT_DIR / "ckpt.pt"
torch.save({
    "state_dict": model.state_dict(),
    "config": cfg.__dict__,
    "vocab_size": len(vocab),
    "stages": [f.name for f in DATA_FILES],
    "hyper": {"MAX_LEN": MAX_LEN, "BATCH": BATCH, "EPOCHS": EPOCHS,
              "LR": LR, "WD": WD, "SEED": SEED},
}, ckpt_path)
print(f"saved → {ckpt_path}  ({ckpt_path.stat().st_size/1e6:.1f} MB)")

## 8 · Evaluate — yes/no accuracy on syllogisms

Conditional log-likelihood: for each held-out syllogism, score the two
candidate answer continuations and pick the argmax.

Answer token verified against our vocab:

| lang | yes | no |
|------|-----|----|
| en   | `[BOS] LIT:yes [EOS]`  | `[BOS] REL:neg [EOS]` |
| ar   | `[BOS] LIT:نعم [EOS]` | `[BOS] STR:neg:general [EOS]` |

In [ ]:
from collections import defaultdict

ANSWER_CANDIDATES = {
    "en": {"yes": ["[BOS]", "LIT:yes", "[EOS]"],
           "no":  ["[BOS]", "REL:neg",  "[EOS]"]},
    "ar": {"yes": ["[BOS]", "LIT:نعم", "[EOS]"],
           "no":  ["[BOS]", "STR:neg:general", "[EOS]"]},
}

SYLLOG_PATH = DATA_DIR / "stage-2b-syllogisms.tokenized.jsonl"
EVAL_MAX = 2000     # cap for speed

records = [json.loads(line) for line in open(SYLLOG_PATH, encoding="utf-8")]
_, val_idx = split_indices(len(records), val_frac=0.05, seed=SEED)
eval_recs = [records[i] for i in val_idx[:EVAL_MAX]]
print(f"eval on {len(eval_recs)} records")

cand_ids = {
    lang: {k: ids_from_tokens(v, vocab) for k, v in cands.items()}
    for lang, cands in ANSWER_CANDIDATES.items()
}

def prefix_tokens(rec):
    seq = list(rec["question_tokens"]["reasoning"])
    for step in rec["cot_tokens"]:
        seq.extend(step["reasoning"])
    return seq

@torch.no_grad()
def score(prefix, cand):
    seq = prefix + cand
    if len(seq) > MAX_LEN:
        seq = seq[-MAX_LEN:]
    n = min(len(cand), len(seq))
    ids = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)
    logits = model(ids)
    logp = F.log_softmax(logits, dim=-1)
    T = ids.size(1)
    total = 0.0
    for off in range(n):
        t = T - n + off
        prev = t - 1
        if prev < 0: continue
        total += logp[0, prev, ids[0, t]].item()
    return total

model.eval()
correct = 0; total = 0
by_lang = defaultdict(lambda: [0, 0])
by_val  = defaultdict(lambda: [0, 0])
by_diff = defaultdict(lambda: [0, 0])

for rec in tqdm(eval_recs, desc="eval"):
    lang = rec["lang"]
    if lang not in cand_ids: continue
    gold = str(rec["answer"]).strip()
    gold_label = "yes" if gold in {"yes", "نعم"} else "no" if gold in {"no", "لا"} else None
    if gold_label is None: continue

    prefix = ids_from_tokens(prefix_tokens(rec), vocab)
    scores = {k: score(prefix, v) for k, v in cand_ids[lang].items()}
    pred = max(scores, key=scores.get)

    ok = int(pred == gold_label)
    correct += ok; total += 1
    by_lang[lang][0] += ok; by_lang[lang][1] += 1
    vkey = "valid" if gold_label == "yes" else "invalid"
    by_val[vkey][0] += ok; by_val[vkey][1] += 1
    d = rec.get("meta", {}).get("difficulty", "?")
    by_diff[d][0] += ok; by_diff[d][1] += 1

def _fmt(d):
    return {k: {"acc": round(v[0]/v[1], 4) if v[1] else 0.0, "n": v[1]} for k, v in d.items()}

summary = {
    "accuracy": round(correct / max(1, total), 4),
    "n": total,
    "by_lang": _fmt(by_lang),
    "by_validity": _fmt(by_val),
    "by_difficulty": _fmt(by_diff),
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
(OUT_DIR / "eval.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

## 9 · Summary

Run artifacts saved to Drive at `OUT_DIR`:

- `ckpt.pt` — full final checkpoint (load with `torch.load(...)` + rebuild `TinyGPT`)
- `ckpt-epoch{N}.pt` — per-epoch snapshots
- `train_log.jsonl` — step-level training curve
- `eval.json` — held-out yes/no accuracy

**What to look for:**

- `val_ppl` should fall steadily across epochs. First epoch usually lands in the 1.5-3× range, bottoming out near 1.1-1.3 for this corpus.
- `accuracy` well above 50% (chance) — otherwise the reasoning-token stream isn't separating the two labels and the projection/tokenizer design needs revisiting.
- `by_lang` EN and AR should be roughly comparable — if EN ≫ AR, the bilingual invariant we claim is weaker than assumed.
- `by_difficulty` `easy > medium > hard` is expected.

Bring the `eval.json` back into the repo when you're done and we can discuss next steps.